# 🚀 Day 28 Lab: Kaggle GPU Serving & Ngrok Tunnels
### Hướng dẫn thiết lập môi trường:
1. Truy cập cài đặt Notebook bên phải (Settings).
2. Trong mục **Accelerator**, chọn **GPU T4 x2** (hoặc **GPU P100**).
3. Chạy tuần tự các Cell bên dưới.

### Cell 1: Cài đặt các thư viện cần thiết

In [ ]:
# Cài đặt công cụ chạy mô hình vLLM và các cổng API dịch vụ
!pip install -q vllm fastapi uvicorn pyngrok sentence-transformers

### Cell 2: Cấu hình mã thông báo ngrok để expose cổng kết nối
*(Lấy mã token miễn phí tại: https://dashboard.ngrok.com/get-started/your-authtoken)*

In [ ]:
from pyngrok import ngrok
# Thay thế chuỗi dưới đây bằng Ngrok Auth Token cá nhân của bạn
ngrok.set_auth_token("YOUR_NGROK_AUTHTOKEN_HERE")

### Cell 3: Khởi động dịch vụ suy luận vLLM (Mô hình Qwen 7B) & Mở Ngrok Tunnel

In [ ]:
import subprocess, threading, time
from pyngrok import ngrok

# Hàm khởi động vLLM OpenAI-compatible server trên cổng 8001
def run_vllm():
    subprocess.run([
        "python", "-m", "vllm.entrypoints.openai.api_server",
        "--model", "Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4",
        "--port", "8001",
        "--max-model-len", "4096",
        "--gpu-memory-utilization", "0.85"
    ])

# Chạy vLLM trong luồng phụ (background thread)
print("Đang tải mô hình Qwen 7B vào GPU...")
thread = threading.Thread(target=run_vllm, daemon=True)
thread.start()

# Chờ vLLM load xong model (tầm 1.5 - 2 phút)
time.sleep(90)

# Mở cổng ngrok 8001 cho vLLM
vllm_tunnel = ngrok.connect(8001, "http")
print("\n" + "="*50)
print(f"🔥 ĐƯỜNG DẪN vLLM URL (Hãy copy URL này dán vào VLLM_NGROK_URL ở local):")
print(vllm_tunnel.public_url)
print("="*50)

### Cell 4: Khởi động FastAPI Embedding Dịch Vụ & Mở Ngrok Tunnel (Port 8002)

In [ ]:
from fastapi import FastAPI
from sentence_transformers import SentenceTransformer
import uvicorn, threading
from pyngrok import ngrok

# Khởi tạo API Server sinh vector nhúng bằng mô hình BGE
app = FastAPI()
model = SentenceTransformer("BAAI/bge-small-en-v1.5")

@app.post("/embed")
def embed(data: dict):
    texts = data["texts"]
    embeddings = model.encode(texts).tolist()
    return {"embeddings": embeddings}

# Khởi chạy uvicorn server ở background trên cổng 8002
def run_embed():
    uvicorn.run(app, host="0.0.0.0", port=8002)

threading.Thread(target=run_embed, daemon=True).start()

# Mở cổng ngrok 8002 cho dịch vụ nhúng
embed_tunnel = ngrok.connect(8002, "http")
print("\n" + "="*50)
print(f"✨ ĐƯỜNG DẪN EMBEDDING URL (Hãy copy URL này dán vào EMBED_NGROK_URL ở local):")
print(embed_tunnel.public_url)
print("="*50)